In [ ]:
import time
import random
import os
import wfdb
import idx2numpy
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.io as sio

from pathlib import Path
from skimage import draw
from datetime import datetime
from math import log10, sqrt
from screenlib import util, model, optimizer, screening
from matplotlib import style
from matplotlib.ticker import LogLocator, FixedLocator, ScalarFormatter
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [ ]:
sys.path.append(str(Path().resolve().parent))

import source.optimizer as optimizer
import source.model as model

In [ ]:
plt.style.use('default')
plt.rcParams.update({'font.size': 16})
plt.rcParams.update({'font.family': 'serif'})

x_ticks = [0.01,0.05,0.1,0.5,1.0]

In [ ]:
np.random.seed(5)
random.seed(5)

# Loading dataset

In [ ]:
def normalize_to_minus1_1(X, X_min, X_max):
    return 2*(X-X_min)/(X_max-X_min) - 1

def denormalize_from_minus1_1(X_norm, X_min, X_max):
    return 0.5*(X_max-X_min)*(X_norm+1) + X_min

In [ ]:
num_samples = 8760
sigma = 1.6e-1

In [ ]:
data = pd.read_excel('../datasets/air-quality/AirQualityUCI.xlsx')

In [ ]:
data.replace(-200, np.nan, inplace=True)
data.interpolate(method='linear', inplace=True)

data.fillna(method='ffill', inplace=True)
data.fillna(method='bfill', inplace=True)

In [ ]:
t_denorm = data['T'].T
t_min    = min(t_denorm)
t_max    = max(t_denorm)
t        = normalize_to_minus1_1(t_denorm, t_min, t_max)

In [ ]:
rh_denorm = data['RH'].T
rh_min  = min(rh_denorm)
rh_max  = max(rh_denorm)
rh      = normalize_to_minus1_1(rh_denorm, rh_min, rh_max)

In [ ]:
ah_denorm = data['AH'].T
ah_min    = min(ah_denorm)
ah_max    = max(ah_denorm)
ah        = normalize_to_minus1_1(ah_denorm, ah_min, ah_max)

In [ ]:
t_denorm_1y  = np.array(t_denorm[0:num_samples:1])
rh_denorm_1y = np.array(rh_denorm[0:num_samples:1])
ah_denorm_1y = np.array(ah_denorm[0:num_samples:1])

Y_denorm  = np.column_stack((t_denorm_1y, rh_denorm_1y, ah_denorm_1y))

In [ ]:
t_1y  = np.array(t[0:num_samples:1])
rh_1y = np.array(rh[0:num_samples:1])
ah_1y = np.array(ah[0:num_samples:1])

Y_teo  = np.column_stack((t_1y, rh_1y, ah_1y))

In [ ]:
X = util.idct_matrix(Y_teo.shape[0])
Y = Y_teo + np.random.normal(0, sigma, Y_teo.shape)

# Experiment

In [ ]:
num_loops     = 10
max_iter      = 100
tol           = 1e-16
eps           = np.finfo(float).eps
n_bins        = 128
n_points      = 20
mag_ord_start = -2
mag_ord_end   = 0

In [ ]:
lmb_r = np.float32(np.logspace(mag_ord_start, mag_ord_end, num=n_points, endpoint=False))

In [ ]:
lmb_max = np.linalg.norm(np.linalg.norm(np.matmul(X.T,Y), axis=1, ord=2), axis=0, ord=np.inf)

In [ ]:
lmb_vec = lmb_r*lmb_max

In [ ]:
params_dict    = {}
functions_dict = {}
screening_dict = {}

In [ ]:
time_min_vec  = []
time_mean_vec = []
time_max_vec  = []
card_vec      = []
true_card_vec = []
supp_mat      = []
error         = []

In [ ]:
params_dict['ss']        = 0
params_dict['alpha']     = 5e-2
params_dict['max_iter']  = max_iter
params_dict['tol']       = tol
params_dict['c']         = 1
params_dict['k_dyn']     = 0
params_dict['k_skip']    = 2
params_dict['stop_crit'] = 1

In [ ]:
functions_dict['f']       = model.linregloss
functions_dict['g']       = model.l12norm
functions_dict['df_du']   = model.grad_linregloss
functions_dict['prox_g']  = model.prox_l12norm
functions_dict['fc']      = model.linregloss_conj
functions_dict['L']       = 1
functions_dict['lmb_max'] = lmb_max

In [ ]:
U_0 = np.zeros([X.shape[1], Y.shape[1]], dtype=np.float32)

In [ ]:
idx_global = np.arange(U_0 .shape[0])

In [ ]:
alphas_dict = {}

alphas_dict['alpha_0'] = 0.85
alphas_dict['alpha_1'] = 0.85
alphas_dict['alpha_2'] = 0.85
alphas_dict['alpha_3'] = 0.85
alphas_dict['alpha_4'] = 0.85

alphas_dict['skew_lim'] = 1e-2

## Vanilla

In [ ]:
params_dict['k_screening'] = 0

screening_dict['mode'] = 0
screening_dict['n_bins'] = n_bins
screening_dict['thresholding'] = 0

for lmb in lmb_vec:

  time_proc = np.zeros(num_loops, dtype=int)

  for k in range(num_loops):
    t_start = datetime.now()
    U_opt , _, meas, fun_costs = optimizer.apg_opt(Y, X, lmb, U_0, params_dict, functions_dict, screening_dict, alphas_dict)
    t_end = datetime.now()
    time_delta = t_end - t_start
    time_proc[k] = time_delta.seconds*1e+6 + time_delta.microseconds

  time_min  = np.min(time_proc)*1e-6
  time_mean = np.mean(time_proc)*1e-6
  time_max  = np.max(time_proc)*1e-6

  time_min_vec.append(time_min)
  time_mean_vec.append(time_mean)
  time_max_vec.append(time_max)

  print("Elapsed time:", "%.2f" % time_mean, "seconds")

  idx_set = idx_global[np.sum(U_opt **2, axis=1)**0.5 > 0]

  supp_mat.append(idx_set)
  card_vec.append(len(idx_set)/U_0 .shape[0])

  Y_est = np.matmul(X,U_opt)

  t_est_1y  = denormalize_from_minus1_1(Y_est[:,0], t_min,  t_max)
  rh_est_1y = denormalize_from_minus1_1(Y_est[:,1], rh_min, rh_max)
  ah_est_1y = denormalize_from_minus1_1(Y_est[:,2], ah_min, ah_max)

  Y_denorm_est  = np.column_stack((t_est_1y, rh_est_1y, ah_est_1y))

  error.append(np.linalg.norm(Y_denorm - Y_denorm_est)/np.linalg.norm(Y_denorm))

# Screening

In [ ]:
screening_dict['thresholding'] = 0
params_dict['k_screening'] = 1

## Screening: Gap-safe ball

In [ ]:
screening_dict['mode'] = 0
screening_dict['n_bins'] = n_bins

for lmb in lmb_vec:

  time_proc = np.zeros(num_loops, dtype=int)

  for k in range(num_loops):
    t_start = datetime.now()
    U_opt , idx_set, meas, fun_costs = optimizer.apg_opt(Y, X, lmb, U_0, params_dict, functions_dict, screening_dict, alphas_dict)
    t_end = datetime.now()
    time_delta = t_end - t_start
    time_proc[k] = time_delta.seconds*1e+6 + time_delta.microseconds

  time_min  = np.min(time_proc)*1e-6
  time_mean = np.mean(time_proc)*1e-6
  time_max  = np.max(time_proc)*1e-6

  time_min_vec.append(time_min)
  time_mean_vec.append(time_mean)
  time_max_vec.append(time_max)

  print("Elapsed time:", "%.2f" % time_mean, "seconds")

  supp_mat.append(idx_set)
  card_vec.append(len(idx_set)/U_0 .shape[0])

  Y_est = np.matmul(X,U_opt)

  t_est_1y  = denormalize_from_minus1_1(Y_est[:,0], t_min,  t_max)
  rh_est_1y = denormalize_from_minus1_1(Y_est[:,1], rh_min, rh_max)
  ah_est_1y = denormalize_from_minus1_1(Y_est[:,2], ah_min, ah_max)

  Y_denorm_est  = np.column_stack((t_est_1y, rh_est_1y, ah_est_1y))

  error.append(np.linalg.norm(Y_denorm - Y_denorm_est)/np.linalg.norm(Y_denorm))

## Screening: Static strong rules

In [ ]:
screening_dict['mode'] = 3
screening_dict['n_bins'] = n_bins

for lmb in lmb_vec:

  time_proc = np.zeros(num_loops, dtype=int)

  for k in range(num_loops):
    t_start = datetime.now()
    U_opt , idx_set, meas, fun_costs = optimizer.apg_opt(Y, X, lmb, U_0, params_dict, functions_dict, screening_dict, alphas_dict)
    t_end = datetime.now()
    time_delta = t_end - t_start
    time_proc[k] = time_delta.seconds*1e+6 + time_delta.microseconds

  time_min  = np.min(time_proc)*1e-6
  time_mean = np.mean(time_proc)*1e-6
  time_max  = np.max(time_proc)*1e-6

  time_min_vec.append(time_min)
  time_mean_vec.append(time_mean)
  time_max_vec.append(time_max)

  print("Elapsed time:", "%.2f" % time_mean, "seconds")

  supp_mat.append(idx_set)
  card_vec.append(len(idx_set)/U_0 .shape[0])

  Y_est = np.matmul(X,U_opt)

  t_est_1y  = denormalize_from_minus1_1(Y_est[:,0], t_min,  t_max)
  rh_est_1y = denormalize_from_minus1_1(Y_est[:,1], rh_min, rh_max)
  ah_est_1y = denormalize_from_minus1_1(Y_est[:,2], ah_min, ah_max)

  Y_denorm_est  = np.column_stack((t_est_1y, rh_est_1y, ah_est_1y))

  error.append(np.linalg.norm(Y_denorm - Y_denorm_est)/np.linalg.norm(Y_denorm))

# Proposed: Adaptive thresholding

In [ ]:
screening_dict['thresholding'] = 1
params_dict['k_screening'] = 1

## Screening: Gap-safe ball + thresholding

In [ ]:
screening_dict['mode'] = 0
screening_dict['n_bins'] = n_bins

for lmb in lmb_vec:

  time_proc = np.zeros(num_loops, dtype=int)

  for k in range(num_loops):
    t_start = datetime.now()
    U_opt , idx_set, meas, fun_costs = optimizer.apg_opt(Y, X, lmb, U_0, params_dict, functions_dict, screening_dict, alphas_dict)
    t_end = datetime.now()
    time_delta = t_end - t_start
    time_proc[k] = time_delta.seconds*1e+6 + time_delta.microseconds

  time_min  = np.min(time_proc)*1e-6
  time_mean = np.mean(time_proc)*1e-6
  time_max  = np.max(time_proc)*1e-6

  time_min_vec.append(time_min)
  time_mean_vec.append(time_mean)
  time_max_vec.append(time_max)

  print("Elapsed time:", "%.2f" % time_mean, "seconds")

  supp_mat.append(idx_set)
  card_vec.append(len(idx_set)/U_0 .shape[0])

  Y_est = np.matmul(X,U_opt)

  t_est_1y  = denormalize_from_minus1_1(Y_est[:,0], t_min,  t_max)
  rh_est_1y = denormalize_from_minus1_1(Y_est[:,1], rh_min, rh_max)
  ah_est_1y = denormalize_from_minus1_1(Y_est[:,2], ah_min, ah_max)

  Y_denorm_est  = np.column_stack((t_est_1y, rh_est_1y, ah_est_1y))

  error.append(np.linalg.norm(Y_denorm - Y_denorm_est)/np.linalg.norm(Y_denorm))

# Results

In [ ]:
os.makedirs("../results", exist_ok=True)

time_mat = np.reshape(time_mean_vec, [4,lmb_vec.shape[0]])
card_mat = np.reshape(card_vec, [4,lmb_vec.shape[0]])

red_gsafe_vec = card_mat[1,:] < 1
red_strong_vec = card_mat[2,:] < 1

error_mat = np.reshape(error, [4,lmb_vec.shape[0]])

## Cardinality

In [ ]:
y_lim = (0,100)
x_lim = (np.min(lmb_r),np.max(lmb_r))

fig, ax = plt.subplots(figsize=(4.5,4))

plt.semilogx(lmb_r[red_gsafe_vec], 100*card_mat[1,red_gsafe_vec], color='g', label='static-gapsafe', marker='^', linewidth=2, clip_on=False)
plt.semilogx(lmb_r[red_strong_vec], 100*card_mat[2,red_strong_vec], color='b', label='static-strong', marker='+', linewidth=2, clip_on=False)
plt.semilogx(lmb_r, 100*card_mat[3,:], color='c', label='proposed', marker='x', linewidth=2, clip_on=False)

plt.grid(which='major', linestyle='--', linewidth=0.75, alpha=0.8)
plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
plt.minorticks_on()

ax = plt.gca()

ax.xaxis.set_major_locator(FixedLocator(x_ticks))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=10))

plt.xlim(x_lim)
plt.ylim(y_lim)

plt.legend(fancybox=True, shadow=True)
plt.xlabel(r"$\frac{\lambda}{\lambda_{max}}$")
plt.ylabel("cardinality [%]")

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.tight_layout()
plt.show()

fig.savefig('../results/env-cardinality.eps', format='eps', bbox_inches='tight')

## Processing time

In [ ]:
y_lim = (0,35)
x_lim = (np.min(lmb_r),np.max(lmb_r))

fig, ax = plt.subplots(figsize=(4.5,4))

plt.semilogx(lmb_r, time_mat[0,:], color='r', label='vanilla', marker='o', linewidth=2, clip_on=False)
plt.semilogx(lmb_r[red_gsafe_vec], time_mat[1,red_gsafe_vec], color='g', label='static-gapsafe', marker='^', linewidth=2, clip_on=False)
plt.semilogx(lmb_r[red_strong_vec], time_mat[2,red_strong_vec], color='b', label='static-strong', marker='+', linewidth=2, clip_on=False)
plt.semilogx(lmb_r, time_mat[3,:], color='c', label='proposed', marker='x', linewidth=2, clip_on=False)

plt.grid(which='major', linestyle='--', linewidth=0.75, alpha=0.8)
plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
plt.minorticks_on()

ax = plt.gca()

ax.xaxis.set_major_locator(FixedLocator(x_ticks))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=10))

plt.xlim(x_lim)
plt.ylim(y_lim)

plt.legend(fancybox=True, shadow=True)
plt.xlabel(r"$\frac{\lambda}{\lambda_{max}}$")
plt.ylabel("time [s]")

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.tight_layout()
plt.show()

fig.savefig('../results/env-time.eps', format='eps', bbox_inches='tight')

## Speedup

In [ ]:
y_lim = (1,50)
x_lim = (np.min(lmb_r),np.max(lmb_r))

fig, ax = plt.subplots(figsize=(4.5,4))

plt.semilogx(lmb_r[red_gsafe_vec], (time_mat[0,red_gsafe_vec]/time_mat[1,red_gsafe_vec]), color='g', label='static-gapsafe', marker='^', linewidth=2, clip_on=False)
plt.semilogx(lmb_r[red_strong_vec], (time_mat[0,red_strong_vec]/time_mat[2,red_strong_vec]), color='b', label='static-strong', marker='+', linewidth=2, clip_on=False)
plt.semilogx(lmb_r, (time_mat[0,:]/time_mat[3,:]), color='c', label='proposed', marker='x', linewidth=2, clip_on=False)

plt.grid(which='major', linestyle='--', linewidth=0.75, alpha=0.8)
plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
plt.minorticks_on()

ax = plt.gca()

ax.xaxis.set_major_locator(FixedLocator(x_ticks))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=10))

plt.xlim(x_lim)
plt.ylim(y_lim)

plt.legend(fancybox=True, shadow=True)
plt.xlabel(r"$\frac{\lambda}{\lambda_{max}}$")
plt.ylabel("speedup")

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.tight_layout()
plt.show()

fig.savefig('../results/env-speedup.eps', format='eps', bbox_inches='tight')

## PSNR

In [ ]:
y_lim = (0,100)
x_lim = (np.min(lmb_r),np.max(lmb_r))

fig, ax = plt.subplots(figsize=(4.5,4))

plt.semilogx(lmb_r, 100*error_mat[0,:], color='m', linewidth=2, clip_on=False)
plt.semilogx(lmb_r, 100*error_mat[1,:], color='m', linewidth=2, clip_on=False)
plt.semilogx(lmb_r, 100*error_mat[2,:], color='m', linewidth=2, clip_on=False)
plt.semilogx(lmb_r, 100*error_mat[3,:], color='m', linewidth=2, clip_on=False)

plt.grid(which='major', linestyle='--', linewidth=0.75, alpha=0.8)
plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
plt.minorticks_on()

ax = plt.gca()

ax.xaxis.set_major_locator(FixedLocator(x_ticks))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=10))

plt.xlim(x_lim)
plt.ylim(y_lim)

plt.legend()
plt.xlabel(r"$\frac{\lambda}{\lambda_{max}}$")
plt.ylabel("error [%]")

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.tight_layout()
plt.show()

fig.savefig('../results/env-error.eps', format='eps', bbox_inches='tight')